# 01 · Reconstrucción de la serie S4 (Fix0 → Fix4)
Misma lógica del notebook `PRONOSTICO_S4_documentado_05102026_1212`, ahora importada desde `s4.reconstruction`. Salida: `data/processed/df_max_s4_reconstruido.csv` con las columnas de bandera `Fix0…Fix_Pico`, que la evaluación usa para **excluir los minutos reconstruidos** de las métricas.

In [ ]:
import sys, warnings
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from s4.config import cargar_config, variante, ruta
from s4.reconstruction import *
cfg = cargar_config('../config/ablacion_2025_2026.yaml')

In [ ]:
RUTA_CRUDO = ruta(cfg, "base") / "data/raw/df_max_s4_all_jic.csv"   # <-- ajustar
df = CargadorDatosS4(str(RUTA_CRUDO), montar_drive=False).cargar()
resumen_crudo = cobertura_diaria(df)
print(resumen_dias_sobre_umbral(resumen_crudo, 90))

In [ ]:
auditor0 = AuditorEventosS4(ConfigAuditoria())
episodios0, _ = auditor0.auditar(df)
df_fix0, bitacora0 = corregir_picos_aislados(df, episodios0, modo="media_local", flag_col="Fix0")
res = ejecutar_pipeline_s4(df_fix0, max_gap=180)
df_final = res["df4"]

In [ ]:
# Auditoría final y corrección de picos que pudieran quedar (Fix_Pico)
auditor = AuditorEventosS4(ConfigAuditoria())
episodios, _ = auditor.auditar(df_final)
df_final_auditado, bitacora = corregir_picos_aislados(df_final, episodios, modo="media_local", flag_col="Fix_Pico")
# OJO: el notebook anterior guardaba df_final (sin Fix_Pico). Aquí se guarda la versión auditada.
salida = ruta(cfg, "s4_reconstruido"); salida.parent.mkdir(parents=True, exist_ok=True)
df_final_auditado.to_csv(salida)
print("Guardado:", salida, f"{len(df_final_auditado):,} filas")